(Tutorial_Get_v2)=
# molsysmt.basic.get
*Retrieving attribute values from a molecular system.*

```{admonition} API documentation
Follow this link for a detailed description of the input arguments, raised errors, and returned objects of this function: {func}`molsysmt.basic.get`.
```

## The Essentials

The `get()` function is designed to turn molecular systems into raw data for your Python scripts. You request available attributes by setting their names to `True` in the arguments. The same call works across supported forms: when a form does not expose a cheap direct getter, MolSysMT follows that form's declared conversion route transparently.

In [1]:
import molsysmt as msm
from molsysmt import systems

lysozyme = systems['T4 lysozyme L99A']['181l.bcif.gz']

# Get the name of the first 5 groups (residues)
group_names = msm.get(lysozyme, element='group', selection=[0,1,2,3,4], group_name=True)
print(f"Group names: {group_names}")

Group names: ['MET', 'ASN', 'ILE', 'PHE', 'GLU']


## Data Integrity & Units

When you retrieve physical properties (like coordinates, times, or masses), MolSysMT returns **quantities with units** to ensure physical consistency.

```{tip}
MolSysMT uses the **PyUnitWizard** suite to handle units. To learn how to extract numeric values or convert units, see the [**Governance: Unit Safety**](../../foundations/06_governance/quantities_and_units.ipynb) section.
```

In [2]:
# Get the coordinates of the first atom
coords = msm.get(lysozyme, selection=0, coordinates=True)

print(f"Coordinates: {coords}")
print(f"Data type: {type(coords)}")

Coordinates: [[[4.398199999999999 -0.3258 0.9163]]] nanometer
Data type: <class 'pint.Quantity'>


## Reading chemical-state attributes

Native topology objects store isotope as stable atom metadata. Formal charge, atom aromaticity, radical state, implicit-hydrogen semantics, stereochemistry, components, and rich bond fields belong to chemical states. Rich bond fields keep integral order, fractional order, relationship type, aromaticity, conjugation, stereochemistry and its reference atoms, direction, component participation, and evidence independent. Use `chemical_state='reference'` (the default), `chemical_state='structure'`, or a 0-based integer index to resolve state information. The `'structure'` mode uses `structure_indices` and requires their nullable `structure_chemical_state_index` values to identify one state. State IDs are not selectors because they need not be unique. Missing nullable values are returned as `pandas.NA`. Formal charge is an integer count in elementary-charge units.

In [3]:
from molsysmt.native import Topology
topology = Topology(n_atoms=3)
msm.set(topology, element='atom', chemical_state=0, formal_charge=[0, -1, 1])
charges = msm.get(topology, element='atom', chemical_state=0, formal_charge=True)
charged_atoms = msm.select(topology, selection='formal_charge != 0', chemical_state=0)
print(charges, charged_atoms)

[0, -1, 1] [1, 2]


State metadata is requested from `element='system'`. These inventory attributes describe every available state and are not filtered when `chemical_state` selects one state for another attribute.

In [4]:
msm.get(
    topology, element='system', output_type='dictionary',
    chemical_state_index=True, chemical_state_id=True,
    n_chemical_states=True, reference_chemical_state_index=True,
    connectivity_completeness=True, component_completeness=True,
    component_evidence=True,
)

{'chemical_state_index': [0],
 'chemical_state_id': ['None'],
 'n_chemical_states': 1,
 'reference_chemical_state_index': 0,
 'connectivity_completeness': ['unavailable'],
 'component_completeness': ['unavailable'],
 'component_evidence': ['unknown']}

## Derived Attributes

Some attributes are calculated from data already present in the molecular form. For example, MolSysMT derives box lengths, angles, shape, and volume from the box matrix without requiring every form adapter to implement those calculations separately. Missing box data produces `None`.

In [5]:
box_lengths, box_angles, box_volume = msm.get(
    lysozyme,
    element='system',
    box_lengths=True,
    box_angles=True,
    box_volume=True,
)
print(box_lengths, box_angles, box_volume)

[[6.09 6.09 9.7]] nanometer [[1.570796 1.570796 2.094395]] radian [311.55659621309997] nanometer ** 3


## Multiple Attributes

If you request more than one attribute, `get()` returns a list of values in the same order as requested.

In [6]:
n_atoms, n_groups = msm.get(lysozyme, n_atoms=True, n_groups=True)
print(f"Total atoms: {n_atoms} | Total groups: {n_groups}")

Total atoms: 1441 | Total groups: 302


### Combining atom and system attributes

A single call can combine attributes from different element levels when the catalog identifies one unambiguous level for an otherwise incompatible attribute. For example, with `element='atom'`, an atom selection applies to `coordinates`, while `structure_id` is evaluated for the system and is unaffected by that selection. Values and dictionary entries retain the order in which the attributes were requested.

In [7]:
coordinates, structure_id = msm.get(
    lysozyme,
    element='atom',
    selection=[0, 1],
    coordinates=True,
    structure_id=True,
)

### Thermodynamic structure series

Native `Structures` objects, the `molsysmt.StructuresDict` form, and the public `molsysmt.MolSys` container can retain `temperature`, `potential_energy`, and `kinetic_energy` for each structure. Requesting `total_energy=True` returns their sum when both energy components are available. `StructuresDict` keeps these optional series through conversion and respects repeated or non-monotonic `structure_indices`. A `molsysviewer.MolSysView` preserves the values through its attached molecular system, whereas `nglview.NGLWidget` does not claim them because its public representation does not retain them.

--- 

```{key-takeaway}
`molsysmt.basic.get` is the gateway to your data. Use it to feed coordinates and metadata into your custom analysis pipelines with total unit safety.
```

```{seealso}
{ref}`Tutorial_Select_v2`: How to filter the elements used in `get()`.